<a href="https://colab.research.google.com/github/jossan-sj8/PSS-C-UOH/blob/main/ejercicios_clase_08092026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejercicio 1: Evaluación Recursiva de Expresiones en Árbol Binario

Escriba un programa o conjunto de funciones en C que implemente una calculadora evaluadora de expresiones representadas mediante un árbol binario. Para esto, asuma que la estructura de cada nodo del árbol almacena un operador (por ejemplo '+' o '*') o bien un operando numérico cuando se trata de una hoja (indicándolo con '\0').

Su solución debe:

Incluir una función constructora para crear y asignar memoria dinámicamente en el Heap a los nodos del árbol.

Implementar una función que reciba el puntero a la raíz del árbol y calcule recursivamente el resultado entero de evaluar la expresión representada.

In [ ]:
%%writefile program1.c
#include <stdio.h>
#include <stdlib.h>

typedef struct Node{
  char op;
  int val;
  struct Node *left;
  struct Node *right;
}Node;

Node *crear_nodo(char op, int val, Node *l, Node *r){
  Node *n = (Node *)malloc(sizeof(Node));
  if (n == NULL) return NULL;
  n-> op = op;
  n -> val = val;
  n -> left = l;
  n -> right = r;
  return n;
}

int evaluar(Node *root){

  if(root == NULL) return 0;

  if(root -> op == '\0'){
    return root -> val;
  }

int izq = evaluar(root -> left);
int der = evaluar(root -> right);

if (root->op == '+') return izq + der;
if (root->op == '*') return izq * der;

return 0;

}

int main(){

  Node *n3 = crear_nodo('\0',3,NULL,NULL);
  Node *n5 = crear_nodo('\0',5,NULL,NULL);

  Node *suma = crear_nodo('+',0,n3,n5);

  Node *n2 = crear_nodo('\0',2,NULL,NULL);

  Node *raiz = crear_nodo('*',0,suma,n2);

  printf("Resultados de la expresion %d\n", evaluar(raiz));
  return 0;

}

Overwriting program1.c


In [ ]:
!gcc program1.c -o program && ./program

Resultados de la expresion 16


# Ejercicio 2: Parsing y Construcción Dinámica de Árboles desde Cadenas

Dado el tipo de dato Node definido en el Ejercicio 1, se desea procesar expresiones matemáticas en texto (e.g., "3+4*7"). Por simplicidad, asuma que la cadena no contiene espacios en blanco, que está bien formada y que solo contiene operandos de un solo dígito sin paréntesis.

Escriba una función en C int calcular(char *expresion) que tome una cadena de texto representando una expresión aritmética y retorne el valor numérico (entero) de su evaluación.

Para lograr esto, implemente internamente la estrategia de parseo por construcción de árbol binario de expresión: se debe localizar el operador de menor precedencia (+ antes que *) para dividir la cadena recursivamente en sus subárboles izquierdo y derecho, reservar la memoria necesaria para la estructura mediante asignación dinámica, y posteriormente evaluar el árbol resultante.

In [ ]:
%%writefile program2.c
#include <stdio.h>
#include <stdlib.h>

typedef struct Node {
    char op;            // '+', '*' o '\0' si es número
    int val;            // Valor si op == '\0'
    struct Node *left, *right;
} Node;

// 1. Construye el árbol leyendo el string s entre los índices [start, end]
Node* build_tree(char *s, int start, int end) {
    if (start > end) return NULL;

    int op_pos = -1;

    // Buscar primero '+' (menor precedencia, queda más arriba en el árbol)
    for (int i = end; i >= start; i--) {
        if (s[i] == '+') { op_pos = i; break; }
    }
    // Si no hay '+', buscar '*'
    if (op_pos == -1) {
        for (int i = end; i >= start; i--) {
            if (s[i] == '*') { op_pos = i; break; }
        }
    }

    Node *n = (Node *)malloc(sizeof(Node));
    if (op_pos != -1) {
        // Nodo Operador
        n->op = s[op_pos];
        n->left = build_tree(s, start, op_pos - 1);
        n->right = build_tree(s, op_pos + 1, end);
    } else {
        // Nodo Número (Hoja)
        n->op = '\0';
        n->val = s[start] - '0'; // Convierte char a int
        n->left = n->right = NULL;
    }
    return n;
}

// 2. Evalúa numéricamente el árbol generado
int evaluate(Node *root) {
    if (!root) return 0;
    if (root->op == '\0') return root->val;

    if (root->op == '+') return evaluate(root->left) + evaluate(root->right);
    if (root->op == '*') return evaluate(root->left) * evaluate(root->right);
    return 0;
}

// 3. Función principal solicitada
int calcular(char *expresion) {
    int len = 0;
    while (expresion[len] != '\0') len++; // Obtener tamaño

    Node *root = build_tree(expresion, 0, len - 1);
    return evaluate(root);
}

int main() {
    char expr[] = "3+4*7";
    printf("Resultado: %d\n", calcular(expr));
    return 0;
}

Writing program2.c


In [ ]:
!gcc program2.c -o program && ./program

Resultado: 31


# Ejercicio 3: Recorrido y Conteo de Nodos Hoja en un Árbol Binario

Dada la estructura estándar para representar un nodo en un árbol binario de enteros:



In [ ]:
typedef struct Node {
    int val;
    struct Node *left;
    struct Node *right;
} Node;

Escriba una función recursiva int contar_hojas(Node *root) que reciba el puntero a la raíz del árbol y determine la cantidad total de nodos hoja que contiene. Considere que un nodo se define como "hoja" si no posee subárbol izquierdo ni subárbol derecho (left == NULL y right == NULL). La función debe manejar adecuadamente el caso de recibir un árbol vacío (NULL).

In [ ]:
%%writefile program3.c
#include <stdio.h>
#include <stdlib.h>

typedef struct Node {
    int val;
    struct Node *left;
    struct Node *right;
} Node;

// Función auxiliar para construir nodos dinámicamente en el Heap
Node* crear_nodo(int valor) {
    Node *n = (Node *)malloc(sizeof(Node));
    if (n == NULL) return NULL;
    n->val = valor;
    n->left = NULL;
    n->right = NULL;
    return n;
}

// Función recursiva que cuenta los nodos hoja
int contar_hojas(Node *root) {
    // Caso Base 1: Árbol o subárbol vacío
    if (root == NULL) return 0;

    // Caso Base 2: El nodo actual es una hoja (sin hijos)
    if (root->left == NULL && root->right == NULL) return 1;

    // Caso Recursivo: Sumar hojas del subárbol izquierdo y derecho
    return contar_hojas(root->left) + contar_hojas(root->right);
}

// MAIN PARA PRUEBAS
int main() {
    Node *root = crear_nodo(10);
    root->left = crear_nodo(5);
    root->right = crear_nodo(15);
    root->left->left = crear_nodo(2);

    printf("Cantidad total de nodos hoja: %d\n", contar_hojas(root));

    // Liberación de memoria
    free(root->left->left);
    free(root->left);
    free(root->right);
    free(root);

    return 0;
}

Writing program3.c


In [ ]:
!gcc program3.c -o program && ./program

Cantidad total de nodos hoja: 2


# Ejercicio 4: Liberación de Memoria Dinámica en un Árbol Binario

Dado el tipo de dato Node para representar un nodo en un árbol binario de expresión:




In [ ]:
typedef struct Node {
    char op;
    int val;
    struct Node *left;
    struct Node *right;
} Node;

Escriba una función recursiva en C void liberar_arbol(Node *root) que reciba el puntero a la raíz de un árbol binario y libere toda la memoria reservada dinámicamente en el Heap para sus nodos.

Para garantizar la integridad del programa, la función debe asegurar la liberación previa de los subárboles hijo antes de liberar el nodo padre (recorrido Post-Order) y manejar adecuadamente el caso de un árbol vacío (NULL).

In [ ]:
%%writefile program4.c
#include <stdio.h>
#include <stdlib.h>

typedef struct Node {
    char op;            // '+', '*' o '\0' si es un número
    int val;            // Valor numérico si op == '\0'
    struct Node *left;
    struct Node *right;
} Node;

// Función auxiliar para crear nodos en Heap
Node* crear_nodo(char op, int val, Node *l, Node *r) {
    Node *n = (Node *)malloc(sizeof(Node));
    if (n == NULL) return NULL;
    n->op = op;
    n->val = val;
    n->left = l;
    n->right = r;
    return n;
}

// Función solicitada: Liberación recursiva de memoria dinámicamente
void liberar_arbol(Node *root) {
    // Caso Base: Si el árbol o subárbol está vacío, no hay nada que liberar
    if (root == NULL) return;

    // 1. Liberar recursivamente el subárbol izquierdo
    liberar_arbol(root->left);

    // 2. Liberar recursivamente el subárbol derecho
    liberar_arbol(root->right);

    // 3. Liberar la memoria del nodo actual (Padre)
    free(root);
}

// MAIN PARA PRUEBAS
int main() {
    // Construcción manual de la expresión: (3 + 5) * 2
    Node *n3 = crear_nodo('\0', 3, NULL, NULL);
    Node *n5 = crear_nodo('\0', 5, NULL, NULL);
    Node *suma = crear_nodo('+', 0, n3, n5);
    Node *n2 = crear_nodo('\0', 2, NULL, NULL);
    Node *raiz = crear_nodo('*', 0, suma, n2);

    // Liberación de toda la memoria consumida por la estructura
    liberar_arbol(raiz);
    raiz = NULL; // Buena práctica: evitar punteros colgantes

    printf("Memoria liberada exitosamente.\n");
    return 0;
}

Writing program4.c


In [ ]:
!gcc program4.c -o program && ./program

Memoria liberada exitosamente.
